# Exploratory Data Analysis for Machine Learning
## 01 — Reading Data: Game Theme

Notebook ini memperkenalkan cara membaca data game dari **CSV** dan **SQLite**. Semua data adalah data sintetis untuk latihan, bukan data penjualan nyata.

**Tujuan:** membaca file, memeriksa bentuk data, melihat tipe kolom, dan menjalankan query SQL sederhana.

In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_DIR / 'data'

print('Folder proyek:', PROJECT_DIR)
print('Isi folder data:', [p.name for p in DATA_DIR.iterdir()])

Folder proyek: d:\smester 5\Exploratory_Data_Analysis_for_Machine_Learning
Isi folder data: ['games_eda.csv', 'game_platform.db', 'player_reviews.csv']


## 1. Membaca file CSV

In [2]:
games = pd.read_csv(DATA_DIR / 'games_eda.csv')
reviews = pd.read_csv(DATA_DIR / 'player_reviews.csv')

print('Games:', games.shape)
print('Reviews:', reviews.shape)
display(games.head())

Games: (80, 10)
Reviews: (240, 6)


,game_id,title,genre,platform,release_year,studio,rating,price_usd,sales_millions,review_count
0,1,Wild Voyage 01,Strategy,PlayStation,2017,Northstar Studio,6.1,14.99,1.05,1357
1,2,Amber Islands 02,Strategy,Switch,2025,Blue Comet,6.4,14.99,2.51,15922
2,3,Amber Frontier 03,Puzzle,Xbox,2019,Orbit Works,9.1,4.99,6.27,10055
3,4,Cosmic Legends 04,Adventure,Xbox,2016,Maple Byte,8.3,61.99,2.10,3181
4,5,Frost Racers 05,Strategy,Xbox,2020,Orbit Works,5.9,9.99,3.10,6481


## 2. Memahami struktur data

In [3]:
games.info()

display(games.describe(include="number"))

print("Nilai kosong per kolom:")
print(games.isna().sum())


<class 'pandas.DataFrame'>
RangeIndex: 80 entries, 0 to 79
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   game_id         80 non-null     int64  
 1   title           80 non-null     str    
 2   genre           80 non-null     str    
 3   platform        80 non-null     str    
 4   release_year    80 non-null     int64  
 5   studio          80 non-null     str    
 6   rating          80 non-null     float64
 7   price_usd       80 non-null     float64
 8   sales_millions  80 non-null     float64
 9   review_count    80 non-null     int64  
dtypes: float64(3), int64(3), str(4)
memory usage: 6.4 KB


,game_id,release_year,rating,price_usd,sales_millions,review_count
count,80.0000,80.000000,80.000000,80.000000,80.000000,80.00000
mean,40.5000,2019.775000,7.076250,26.903750,4.547375,8414.01250
std,23.2379,3.097467,1.282032,21.386464,4.179013,5577.39112
min,1.0000,2014.000000,3.800000,0.000000,0.450000,53.00000
25%,20.7500,2017.000000,6.300000,9.990000,2.147500,3200.50000
50%,40.5000,2020.000000,6.800000,18.490000,3.260000,8355.50000
75%,60.2500,2022.000000,7.950000,49.990000,5.655000,13775.50000
max,80.0000,2025.000000,10.000000,61.990000,26.980000,17998.00000


Nilai kosong per kolom:
game_id           0
title             0
genre             0
platform          0
release_year      0
studio            0
rating            0
price_usd         0
sales_millions    0
review_count      0
dtype: int64


## 3. Membaca database SQLite

In [4]:
db_path = DATA_DIR / 'game_platform.db'
with sqlite3.connect(db_path) as con:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)
    display(tables)

,name
0,games
1,play_sessions
2,players


In [5]:
with sqlite3.connect(db_path) as con:
    sample_games = pd.read_sql_query('SELECT * FROM games LIMIT 10', con)
    display(sample_games)

,game_id,title,genre,platform,release_year,rating,price_usd
0,1,Wild Voyage 01,Strategy,PlayStation,2017,6.1,14.99
1,2,Amber Islands 02,Strategy,Switch,2025,6.4,14.99
2,3,Amber Frontier 03,Puzzle,Xbox,2019,9.1,4.99
3,4,Cosmic Legends 04,Adventure,Xbox,2016,8.3,61.99
4,5,Frost Racers 05,Strategy,Xbox,2020,5.9,9.99
5,6,Crystal Voyage 06,Action,PC,2021,5.9,9.99
6,7,Forgotten Realm 07,Sports,PC,2023,4.8,31.99
7,8,Solar Labyrinth 08,RPG,Mobile,2022,6.5,59.99
8,9,Golden Labyrinth 09,Adventure,Xbox,2016,7.4,14.99
9,10,Frost Academy 10,Strategy,PlayStation,2023,8.6,0.00


## 4. Menggabungkan informasi sesi bermain dengan katalog game

In [6]:
query = '''
SELECT g.title, g.genre, g.platform, COUNT(s.session_id) AS total_sessions,
       ROUND(AVG(s.minutes_played), 1) AS avg_minutes
FROM games AS g
LEFT JOIN play_sessions AS s ON g.game_id = s.game_id
GROUP BY g.game_id, g.title, g.genre, g.platform
ORDER BY total_sessions DESC
LIMIT 10;
'''
with sqlite3.connect(db_path) as con:
    popular_sessions = pd.read_sql_query(query, con)
display(popular_sessions)

,title,genre,platform,total_sessions,avg_minutes
0,Golden Labyrinth 09,Adventure,Xbox,6,138.5
1,Wild Realm 53,Adventure,PlayStation,6,109.3
2,Crystal Voyage 06,Action,PC,5,116.2
3,Wild Racers 58,RPG,PC,5,108.4
4,Hidden Islands 66,RPG,PlayStation,5,73.8
5,Velvet Voyage 68,RPG,PC,5,106.2
6,Forgotten Islands 80,Action,PC,5,137.4
7,Wild Voyage 01,Strategy,PlayStation,4,177.8
8,Frost Racers 05,Strategy,Xbox,4,121.5
9,Forgotten Realm 07,Sports,PC,4,78.8


## 5. Ringkasan
- `read_csv()` membaca data CSV menjadi DataFrame.
- SQLite menyimpan tabel yang dapat dibaca dengan query SQL.
- `shape`, `head()`, `info()`, dan `describe()` membantu memahami data.

**Latihan:** ubah query SQL untuk menampilkan 10 game dengan rata-rata menit bermain tertinggi.